# ML Pipeline - Universal CV Framework

## 개요
- **목적**: 범용 ML 파이프라인 (분류/회귀)
- **주요 기능**: 
  - UniversalCVWrapper: 모든 sklearn-style 모델 지원
  - Optuna 하이퍼파라미터 최적화
  - Multi-metric 평가 (accuracy, auc, f1, rmse, mae, r2)

## 구조
1. 데이터 로딩
2. 공통 유틸리티 함수
3. Wrapper 클래스
4. Optuna 최적화 함수
5. 실행 및 결과 저장

In [19]:
# 데이터 로딩
from pathlib import Path
import pandas as pd
import skimpy
import torch

# 프로젝트 루트 경로 (현재 노트북 기준)
PROJECT_ROOT = Path(r"C:\SEMIN\Project\ML_FINAL\3.SELECTED")
DATA_DIR = PROJECT_ROOT

# 데이터 파일 로드
test = pd.read_csv(DATA_DIR / "test_selected.csv")
train = pd.read_csv(DATA_DIR / "train_selected.csv")

print(f"✅ 데이터 로드 완료")
print(f"  Train: {train.shape}")
print(f"  Test: {test.shape}")
skimpy.skim(train)

✅ 데이터 로드 완료
  Train: (30000, 252)
  Test: (19995, 251)


╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 30000  │ │ int64       │ 197   │                                                          │
│ │ Number of columns │ 252    │ │ float64     │ 55    │                                                          │
│ └───────────────────┴────────┘ └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┓  │
│ ┃ column          ┃ NA ┃ NA % ┃ mean     ┃ sd     ┃ p0     ┃ p25     ┃ p50      ┃ p75    ┃ p100     ┃ hist   ┃  │
│ ┡━━━━━━━━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━┩  │
│ │ custid          │  0 │    0 │    15000 │   8660 │      0 │    7500 │    15000 │  22500 │    30000 │ ▇▇▇▇▇▇ │  │
│ │ 트래디셔널      │  0 │    0 │   0.4941 │  1.451 │      0 │       0 │        0 │      0 │       35 │   ▇    │  │
│ │ 셔츠            │  0 │    0 │   0.4029 │   1.04 │      0 │       0 │        0 │      0 │       36 │   ▇    │  │
│ │ 스타킹          │  0 │    0 │   0.3542 │ 0.9362 │      0 │       0 │        0 │      0 │       27 │   ▇    │  │
│ │ 란제리          │  0 │    0 │    0.816 │  1.668 │      0 │       0 │        0 │      1 │       27 │   ▇    │  │
│ │ 내셔날          │  0 │    0 │   0.1075 │ 0.5006 │      0 │       0 │        0 │      0 │       12 │   ▇    │  │
│ │ 영캐주얼        │  0 │    0 │   0.9824 │  2.689 │      0 │       0 │        0 │      1 │       76 │   ▇    │  │
│ │ 아동            │  0 │    0 │    0.547 │  2.541 │      0 │       0 │        0 │      0 │      154 │   ▇    │  │
│ │ 골프웨어        │  0 │    0 │   0.4067 │  1.799 │      0 │       0 │        0 │      0 │       73 │   ▇    │  │
│ │ 캐릭터캐주얼    │  0 │    0 │   0.1378 │ 0.6918 │      0 │       0 │        0 │      0 │       29 │   ▇    │  │
│ │ 완구            │  0 │    0 │   0.2102 │ 0.9538 │      0 │       0 │        0 │      0 │       36 │   ▇    │  │
│ │ 아동복          │  0 │    0 │   0.6403 │  2.781 │      0 │       0 │        0 │      0 │       70 │   ▇    │  │
│ │ 청과            │  0 │    0 │   0.4501 │  1.905 │      0 │       0 │        0 │      0 │       89 │   ▇    │  │
│ │ 핸드백          │  0 │    0 │   0.3977 │ 0.9802 │      0 │       0 │        0 │      0 │       17 │   ▇    │  │
│ │ 머플러          │  0 │    0 │   0.0788 │ 0.3643 │      0 │       0 │        0 │      0 │        9 │   ▇    │  │
│ │ 일용잡화        │  0 │    0 │   0.4023 │  1.169 │      0 │       0 │        0 │      0 │       42 │   ▇    │  │
│ │ 남성구두        │  0 │    0 │  0.06623 │ 0.3544 │      0 │       0 │        0 │      0 │       13 │   ▇    │  │
│ │ 골프(국내)      │  0 │    0 │   0.2272 │  1.147 │      0 │       0 │        0 │      0 │       36 │   ▇    │  │
│ │ 진케주얼        │  0 │    0 │   0.3453 │    1.4 │      0 │       0 │        0 │      0 │       34 │   ▇    │  │
│ │ 영트랜드        │  0 │    0 │   0.4419 │  1.811 │      0 │       0 │        0 │      0 │       57 │   ▇    │  │
│ │ 스포츠웨어      │  0 │    0 │   0.3722 │  1.155 │      0 │       0 │        0 │      0 │       29 │   ▇    │  │
│ │ 색조화장품      │  0 │    0 │   0.2219 │ 0.6809 │      0 │       0 │        0 │      0 │       12 │   ▇    │  │
│ │ 유아복          │  0 │    0 │   0.3907 │  1.864 │      0 │       0 │        0 │      0 │       54 │   ▇    │  │
│ │ 스포츠슈즈      │  

---
## 1. 데이터 로딩 및 전역 설정

---
## 2. 공통 유틸리티 함수

In [20]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ====================================================
# 1. 전역 설정 (Global Configuration)
# ====================================================

if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"🔥 GPU 가속 활성화: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = "cpu"
    print("🐢 CPU 모드로 실행")

USER_NAME = "semin"  # 👈 파일명에 들어갈 사용자 이름
OPTUNA_TIMEOUT = 10      # 👈 Optuna 최적화 시간 제한 (초 단위, 예: 600초 = 10분)
OUTPUT_DIR = Path(r"C:\SEMIN\Project\ML_FINAL\4.OOF_SUBMISSION")  # 👈 출력 파일 저장 경로

# 프로젝트별 설정
TARGET = "gender"  # 타겟 변수명
ID_COL = "custid"  # 식별자 변수명
TIME_COL = None  # 시간 변수명 (timesplit 사용시)

# 실험 설정
TASK_TYPE = "classification"  # "classification" or "regression"
METRIC = "roc_auc"  # roc_auc, accuracy, f1, rmse, mae, r2
CV_SPLITS = 5
RANDOM_STATE = 42
CV_STRATEGY = "stratified"    # 👈 "stratified", "group", "kfold", "timesplit"
ENCODING = "target"        # 👈 "label", "target", "catboost"
SCALING = True  # 👈 True/False


# 그룹 K-Fold용 (group 사용시)
GROUP_COL = None # "patient_id" 등 (group일때만 사용)

# 🎛️ 모델 실행 스위치 (True: 실행 / False: 건너뜀)
ACTIVE_MODELS = {
    "LGBM": True,
    "CatBoost": True,
    "XGBoost": True,
    "RF": True,
    "ET": True,
    "MLP": True,
    "Linear": True,
    "TabM": False,
    "RealMLP": True,
}

# 🎮 [NEW] 모델별 디바이스 지정 (cpu / cuda)
# XGBoost, TabM, RealMLP는 'cuda' 추천
# CatBoost, LGBM은 데이터 크기에 따라 'cpu'가 빠를 수 있음
# RF, ET, Linear는 Sklearn 기반이므로 무조건 'cpu'
MODEL_DEVICE_CONFIG = {
    "LGBM": "cpu",         # 에러(bin size) 방지를 위해 CPU 권장
    "CatBoost": "cpu",     # 사용자가 CPU 선호
    "XGBoost": "cuda",     # GPU 가속 효율 좋음
    "RF": "cpu",           # Sklearn은 GPU 미지원
    "ET": "cpu",           # Sklearn은 GPU 미지원
    "MLP": "cpu",          # Sklearn MLP는 CPU
    "Linear": "cpu",
    "TabM": "cuda",        # 딥러닝 (GPU 필수)
    "RealMLP": "cuda",     # 딥러닝 (GPU 필수)
}

# 모델별 Optuna Trials 설정
TRIALS_CONFIG = {
    "LGBM": 10000,
    "CatBoost": 10000,
    "XGBoost": 10000,
    "RF": 10000,
    "ET": 10000,
    "MLP": 10000,
    "Linear": 10000,
    "TabM": 10000, # Deep Learning (SOTA)
    "RealMLP": 10000, # Deep Learning
}

print(f"Task: {TASK_TYPE} | Target: {TARGET} | Metric: {METRIC}")


# ====================================================
# 2. 피처 타입 자동 분리 함수 (Feature Splitter)
# ====================================================
def get_feature_lists(df, target_col=None, id_col=None):
    """
    데이터프레임을 받아서 수치형, 범주형 피처 리스트를 반환합니다.
    Target과 ID 컬럼은 자동으로 제외합니다.
    """
    # 전체 컬럼에서 제외할 것들 (Target, ID)
    ignore_cols = []
    if target_col and target_col in df.columns:
        ignore_cols.append(target_col)
    if id_col and id_col in df.columns:
        ignore_cols.append(id_col)

    # 제외할 컬럼을 뺀 나머지
    features = [c for c in df.columns if c not in ignore_cols]

    # 1. 수치형(Numeric) 탐색
    num_features = df[features].select_dtypes(include=["number"]).columns.tolist()

    # 2. 범주형(Categorical) 탐색
    cat_features = df[features].select_dtypes(include=["object", "category"]).columns.tolist()

    # [방어 로직] 혹시 놓친 컬럼이 있는지 체크
    remaining = [c for c in features if c not in num_features and c not in cat_features]
    if remaining:
        print(f"⚠️ [알림] 특이한 타입의 컬럼 발견 (범주형으로 포함): {remaining}")
        cat_features.extend(remaining)

    return num_features, cat_features


# ====================================================
# 3. 실행 및 검증
# ====================================================
num_features, cat_features = get_feature_lists(train, TARGET, ID_COL)

print(f"🎯 타겟 변수: '{TARGET}'")
print(f"🆔 ID 변수: '{ID_COL}' (학습 제외)")
print("-" * 50)
print(f"🔢 수치형 피처 ({len(num_features)}개): {num_features}")
print(f"🔤 범주형 피처 ({len(cat_features)}개): {cat_features}")
print("-" * 50)
print(f"✅ 총 학습 피처 개수: {len(num_features) + len(cat_features)}개")

🔥 GPU 가속 활성화: NVIDIA GeForce GTX 1660 SUPER
Task: classification | Target: gender | Metric: roc_auc
🎯 타겟 변수: 'gender'
🆔 ID 변수: 'custid' (학습 제외)
--------------------------------------------------
🔢 수치형 피처 (250개): ['트래디셔널', '셔츠', '스타킹', '란제리', '내셔날', '영캐주얼', '아동', '골프웨어', '캐릭터캐주얼', '완구', '아동복', '청과', '핸드백', '머플러', '일용잡화', '남성구두', '골프(국내)', '진케주얼', '영트랜드', '스포츠웨어', '색조화장품', '유아복', '스포츠슈즈', '라이센스', '수입향수', '주방용품', '트레디셔널캐주얼', '여성구두', '수입도자기', '침구', '야채', '내의', '수영복', '과자류', '곡물', '캐릭터', '냉장식품', '단품', '일반조리', '타운웨어', '조미료', '영커리어캐주얼', '베이직캐주얼', '캐릭터캐쥬얼', '용기보증', '국내종합화장품', '수입', 'DC캐주얼', '스카프', '명품', '향수', '생선', '욕실용품', '가방', '아웃도어', '구두행사', '신생아', '양말', '주류', '트래디셔널캐쥬얼', '미확인코너', '미씨캐릭터', '아동특선', '피혁토탈(B2)', '즉석조리', '수입종합화장품', '건식품', '차류', '캐쥬얼구두', '토탈', '문화', '골프(수입)', '진캐쥬얼', '정육', '디자이너', '넥타이', '패션악세사리', '부띠끄', '완구(문화)', '싸롱화', '특정', '행사', '내셔널', '영캐쥬얼', '뉴베이직캐주얼', '하이캐쥬얼', '화장품', '슈즈', '기타식품', '패션란제리', '엘레강스부틱', '테이프', '디자이너숍', '모피니트', '캐리어캐쥬얼', '유니섹스캐주얼', '건강식품', '팬시코너(문화)', '면류', '캐

---
## 3. 인코딩 함수 (Label, Target, OHE)

In [21]:
from sklearn.preprocessing import LabelEncoder
import category_encoders as ce  # pip install category_encoders 필요

# ====================================================
# 3. 인코딩 함수 정의 (Modular Encoders)
# ====================================================


def label_encoding(train_df, test_df, cat_cols):
    """
    [Label Encoding]
    - 범주형 변수를 0, 1, 2... 정수로 변환
    - Tree 모델(LGBM, XGB)에서 가장 무난하고 빠름
    """
    # 원본 오염 방지
    train = train_df.copy()
    test = test_df.copy()

    # Train/Test 범주를 합쳐서 학습 (Unknown 방지)
    # 실제로는 Train에만 fit 하는게 정석이지만, 대회용 베이스라인으론 합치는게 국룰
    for col in cat_cols:
        le = LabelEncoder()
        all_values = pd.concat([train[col], test[col]]).astype(str)
        le.fit(all_values)

        train[col] = le.transform(train[col].astype(str)).astype("category")
        test[col] = le.transform(test[col].astype(str)).astype("category")

    print(f"✅ Label Encoding 완료 ({len(cat_cols)}개 컬럼)")
    return train, test


def target_encoding(train_df, test_df, cat_cols, target_col, smoothing=10):
    """
    [Target Encoding]
    - 범주를 해당 범주의 타겟 평균값으로 변환
    - 주의: 반드시 Data Leakage 방지를 위해 CV Loop 안에서 쓰거나,
      category_encoders 라이브러리의 smoothing 기능을 믿어야 함.
    """
    train = train_df.copy()
    test = test_df.copy()

    # category_encoders 라이브러리 사용 (LeaveOneOut이나 TargetEncoder 추천)
    # 여기서는 가장 기본적인 TargetEncoder 사용
    encoder = ce.TargetEncoder(cols=cat_cols, smoothing=smoothing)

    # Train은 fit_transform, Test는 transform
    # 이때 Train의 y값(정답)을 보고 매핑을 만듦
    train[cat_cols] = encoder.fit_transform(train[cat_cols], train[TARGET])
    test[cat_cols] = encoder.transform(test[cat_cols])

    print(f"✅ Target Encoding 완료 ({len(cat_cols)}개 컬럼)")
    return train, test


def catboost_encoding_process(train_df, test_df, cat_cols):
    """
    [CatBoost Native]
    - 아무것도 안 하고 'category' 타입으로만 변환
    - CatBoost 모델은 내부적으로 이걸 알아서 처리함 (가장 좋음)
    """
    train = train_df.copy()
    test = test_df.copy()

    for col in cat_cols:
        # 문자열(object) 그대로 두거나 category 타입으로만 명시
        train[col] = train[col].astype("category")
        test[col] = test[col].astype("category")

    print(f"✅ CatBoost용 처리 완료 (Raw Category 유지)")
    return train, test


# ====================================================
# 4. 인코딩 통합 실행 함수 (Switcher)
# ====================================================
def apply_encoding(train, test, cat_cols, target_col, method="label"):
    print(f"\n🔄 인코딩 적용 중... (Method: {method})")

    if method == "label":
        return label_encoding(train, test, cat_cols)

    elif method == "target":
        return target_encoding(train, test, cat_cols, target_col)

    elif method == "catboost":
        return catboost_encoding_process(train, test, cat_cols)

    else:
        print("⚠️ 잘못된 인코딩 방식입니다. 원본을 반환합니다.")
        return train, test

## CV설정

In [22]:
"""
Universal CV Pipeline - FINAL VERSION
분류 + 회귀 통합, 모든 개선사항 포함

개선사항:
1. Multi-Metric Tracking (Fold별 + Overall OOF)
2. Target Encoding Leakage 방지 (Smoothing + Holdout)
4. return_scores 옵션
"""

import pandas as pd
import numpy as np
import os
import shutil
import uuid
import category_encoders as ce
from sklearn.model_selection import StratifiedKFold, GroupKFold, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    log_loss,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# ========================================
# Universal CV Wrapper
# ========================================
class UniversalCVWrapper:
    """
    Final Universal CV Wrapper (Refactored for Device Config)
    - 모델별 CPU/GPU 설정 적용 (CatBoost 에러 해결)
    - Target Encoding Smoothing 파라미터화
    """
    
    def __init__(
        self,
        model,
        model_name,  # 👈 [핵심] 모델 이름을 받아서 Config를 조회함
        task_type=TASK_TYPE,
        n_splits=CV_SPLITS,
        random_state=RANDOM_STATE,
        cv_strategy=CV_STRATEGY,
        encoding=ENCODING,
        scaling=SCALING,
        target_col=TARGET,
        group_col=GROUP_COL,
        te_smoothing=10.0,
        scaler_type="standard",
    ):
        self.model = model
        self.model_name = model_name
        self.task_type = task_type
        self.n_splits = n_splits
        self.random_state = random_state
        self.cv_strategy = cv_strategy
        self.encoding = encoding
        self.scaling = scaling
        self.target_col = target_col
        self.group_col = group_col
        self.te_smoothing = te_smoothing
        self.scaler_type = scaler_type

        # 🎮 [핵심] 모델별 디바이스 설정 가져오기 (전역 MODEL_DEVICE_CONFIG 사용)
        # 설정이 없으면 기본적으로 'cpu'로 동작하여 에러 방지
        if 'MODEL_DEVICE_CONFIG' in globals():
            self.device = MODEL_DEVICE_CONFIG.get(self.model_name, "cpu")
        else:
            self.device = "cpu" 

    def _get_splitter(self):
        from sklearn.model_selection import KFold, StratifiedKFold, GroupKFold, TimeSeriesSplit
        if self.task_type == "regression":
            if self.cv_strategy == "group": 
                return GroupKFold(n_splits=self.n_splits)
            elif self.cv_strategy == "timesplit":
                return TimeSeriesSplit(n_splits=self.n_splits)
            else:
                return KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        else:
            if self.cv_strategy == "stratified": 
                return StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
            elif self.cv_strategy == "group": 
                return GroupKFold(n_splits=self.n_splits)
            elif self.cv_strategy == "timesplit":
                return TimeSeriesSplit(n_splits=self.n_splits)
            else:
                return KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)

    def _get_scaler(self):
        from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
        if self.scaler_type == "robust": return RobustScaler()
        elif self.scaler_type == "minmax": return MinMaxScaler()
        return StandardScaler()

    def fit_predict(self, train_df, test_df, return_scores=False):
        # [0] 데이터 준비
        X = train_df.drop(columns=[ID_COL, self.target_col]).copy()
        y = train_df[self.target_col]
        X_test = test_df.drop(columns=[ID_COL]).copy()

        # 스케일러 로그용 텍스트
        scaler_log = self.scaler_type if self.scaling else "None (OFF)"

        print(
            f"🚀 [CV Start] Model: {self.model_name} | Device: {self.device.upper()} | "
            f"Enc: {self.encoding} (sm={self.te_smoothing:.2f}) | Scaler: {scaler_log}"
        )

        cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
        num_features = X.select_dtypes(include=[np.number]).columns.tolist()

        oof_preds = np.zeros(len(X))
        test_preds = np.zeros(len(X_test))
        fold_scores = []

        # [1] 타입 보정
        if cat_features:
            if self.model_name in ["CatBoost", "LGBM", "XGBoost"]:
                X[cat_features] = X[cat_features].astype("category")
                X_test[cat_features] = X_test[cat_features].astype("category")
            else:
                X[cat_features] = X[cat_features].astype(str)
                X_test[cat_features] = X_test[cat_features].astype(str)

        # [Safety] 딥러닝 모델 결측치 0 채움
        if self.model_name in ["TabM", "RealMLP", "MLP"]:
             X[num_features] = X[num_features].fillna(0)
             X_test[num_features] = X_test[num_features].fillna(0)

        # [2] Encoding Setup
        cat_features_for_model = []
        if self.model_name == "CatBoost":
            cat_features_for_model = cat_features
        elif self.encoding == "label" and cat_features:
            for col in cat_features:
                le = LabelEncoder()
                all_values = pd.concat([X[col], X_test[col]]).astype(str)
                le.fit(all_values)
                X[col] = le.transform(X[col].astype(str))
                X_test[col] = le.transform(X_test[col].astype(str))
                if self.model_name == "LGBM":
                    X[col] = X[col].astype("category")
                    X_test[col] = X_test[col].astype("category")
            cat_features_for_model = cat_features

        # [3] CV Loop
        kf = self._get_splitter()
        split_args = [X, y]
        if "group" in self.cv_strategy:
            split_args.append(train_df[self.group_col])

        for fold, (train_idx, val_idx) in enumerate(kf.split(*split_args)):
            X_tr, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            X_te_fold = X_test.copy()

            # [3-1] Target Encoding
            current_cat = cat_features_for_model
            if self.encoding == "target" and cat_features:
                te = ce.TargetEncoder(cols=cat_features, smoothing=self.te_smoothing)
                X_tr[cat_features] = te.fit_transform(X_tr[cat_features], y_tr)
                X_val[cat_features] = te.transform(X_val[cat_features])
                X_te_fold[cat_features] = te.transform(X_te_fold[cat_features])
                current_cat = []

            # [3-2] Scaling
            if self.scaling and num_features:
                scaler = self._get_scaler()
                X_tr[num_features] = scaler.fit_transform(X_tr[num_features])
                X_val[num_features] = scaler.transform(X_val[num_features])
                X_te_fold[num_features] = scaler.transform(X_te_fold[num_features])

            # [4] Model Fitting & Parameters
            # 🚨 [중요] 여기서는 set_params를 호출하지 않습니다! (Optuna 함수에서 Init 할 때 이미 설정됨)
            fit_params = {}
            if self.model_name in ["LGBM", "CatBoost", "XGBoost"]:
                fit_params = {"eval_set": [(X_val, y_val)]}
                
                # LGBM
                if self.model_name == "LGBM":
                    import lightgbm as lgb
                    fit_params["eval_metric"] = "binary_logloss" if self.task_type == "classification" else "rmse"
                    fit_params["callbacks"] = [lgb.early_stopping(50, verbose=False)]
                    if current_cat: fit_params["categorical_feature"] = current_cat

                # CatBoost
                elif self.model_name == "CatBoost":
                    if current_cat: fit_params["cat_features"] = current_cat
                    fit_params["early_stopping_rounds"] = 50
                    fit_params["verbose"] = False

                # XGBoost
                elif self.model_name == "XGBoost":
                    fit_params["verbose"] = False

                self.model.fit(X_tr, y_tr, **fit_params)
            else:
                self.model.fit(X_tr, y_tr)

            # [5] Prediction
            if self.task_type == "classification":
                val_pred = self.model.predict_proba(X_val)[:, 1]
                oof_preds[val_idx] = val_pred
                test_preds += self.model.predict_proba(X_te_fold)[:, 1] / self.n_splits
                
                acc = accuracy_score(y_val, (val_pred >= 0.5).astype(int))
                auc = roc_auc_score(y_val, val_pred)
                print(f"  Fold {fold+1} | Acc: {acc:.4f} | AUC: {auc:.4f}")
                fold_scores.append({"accuracy": acc, "roc_auc": auc})

            else:
                val_pred = self.model.predict(X_val)
                oof_preds[val_idx] = val_pred
                test_preds += self.model.predict(X_te_fold) / self.n_splits
                
                rmse = np.sqrt(mean_squared_error(y_val, val_pred))
                print(f"  Fold {fold+1} | RMSE: {rmse:.4f}")
                fold_scores.append({"rmse": rmse})

            del X_te_fold, X_tr, X_val
            import gc
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        print(f"\n✨ {self.model_name} CV Complete.")
        
        if return_scores:
            return oof_preds, test_preds, fold_scores
        else:
            return oof_preds, test_preds

### 4-2. UniversalCVWrapper
범용 Cross-Validation 래퍼 클래스

---
## 5. Optuna 하이퍼파라미터 최적화
### 설정 확인

In [23]:
# 설정은 최상단에서 이미 정의됨 (TASK_TYPE, METRIC, CV_SPLITS 등)
print(f"\n{'='*60}")
print(f"📊 실험 설정 확인")
print(f"{'='*60}")
print(f"  Task Type    : {TASK_TYPE}")
print(f"  Target       : {TARGET}")
print(f"  Metric       : {METRIC}")
print(f"  CV Splits    : {CV_SPLITS}")
print(f"  Random State : {RANDOM_STATE}")
print(f"\n  Trials Config:")
for model, trials in TRIALS_CONFIG.items():
    print(f"    {model:20s}: {trials}")


📊 실험 설정 확인
  Task Type    : classification
  Target       : gender
  Metric       : roc_auc
  CV Splits    : 5
  Random State : 42

  Trials Config:
    LGBM                : 10000
    CatBoost            : 10000
    XGBoost             : 10000
    RF                  : 10000
    ET                  : 10000
    MLP                 : 10000
    Linear              : 10000
    TabM                : 10000
    RealMLP             : 10000


### 5-1. Import 라이브러리

In [ ]:
# ML 모델 import
from lightgbm import LGBMClassifier, LGBMRegressor
from catboost import CatBoostClassifier, CatBoostRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.linear_model import LogisticRegression, Ridge, Lasso

# Optuna import
import optuna
from optuna.samplers import TPESampler

print("✅ 모델 및 최적화 라이브러리 import 완료")
# %% [4-1] LGBM Optuna
def optimize_lgbm(n_trials=None):
    """LGBM 최적화"""
    if n_trials is None:
        n_trials = TRIALS_CONFIG["LGBM"]
    
    print(f"\n{'='*80}")
    print(f"🔍 LGBM 최적화 ({n_trials} trials)")
    print(f"{'='*80}")
    
    def objective(trial):
        params = {
            # [핵심] 트리 개수와 깊이 대폭 상향
            "n_estimators": trial.suggest_int("n_estimators", 2000, 5000), 
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 32, 256),  # 64 -> 256 (표현력 증대)
            "max_depth": trial.suggest_int("max_depth", 8, 16),      # 깊게 탐색
            
            # 과적합 방지 규제 (범위 넓힘)
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 50.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 50.0, log=True),
            
            "random_state": RANDOM_STATE,
            "verbose": -1,
        }
        # Objective 안에서도 model_name 필수!
        target_device = MODEL_DEVICE_CONFIG["LGBM"]
        params["device"] = "gpu" if target_device == "cuda" else "cpu"

        model = LGBMClassifier(**params) if TASK_TYPE == "classification" else LGBMRegressor(**params)
        wrapper = UniversalCVWrapper(model=model, model_name="LGBM", task_type=TASK_TYPE, n_splits=CV_SPLITS, random_state=RANDOM_STATE, encoding=ENCODING, target_col=TARGET)
        oof_preds, _ = wrapper.fit_predict(train, test)
        
        from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error, r2_score
        
        if TASK_TYPE == "classification":
            if METRIC == "accuracy":
                return accuracy_score(train[TARGET], (oof_preds >= 0.5).astype(int))
            elif METRIC == "roc_auc":
                return roc_auc_score(train[TARGET], oof_preds)
            elif METRIC == "f1":
                return f1_score(train[TARGET], (oof_preds >= 0.5).astype(int))
        else:
            if METRIC == "rmse":
                return np.sqrt(mean_squared_error(train[TARGET], oof_preds))
            elif METRIC == "mae":
                return mean_absolute_error(train[TARGET], oof_preds)
            elif METRIC == "r2":
                return r2_score(train[TARGET], oof_preds)
    
    direction = "maximize" if METRIC in ["accuracy", "roc_auc", "f1", "r2"] else "minimize"
    study = optuna.create_study(direction=direction, sampler=TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials,timeout=OPTUNA_TIMEOUT, show_progress_bar=True)
    
    best_params = study.best_params
    best_params["verbose"] = -1
    print(f"\n✨ Best {METRIC.upper()}: {study.best_value:.4f}")
    
    # [수정됨] 최종 학습 시에도 model_name="LGBM" 전달 필수!
    final_model = LGBMClassifier(**best_params) if TASK_TYPE == "classification" else LGBMRegressor(**best_params)
    wrapper = UniversalCVWrapper(model=final_model, model_name="LGBM", task_type=TASK_TYPE, n_splits=CV_SPLITS, random_state=RANDOM_STATE, encoding=ENCODING, target_col=TARGET)
    oof_preds, test_preds, _ = wrapper.fit_predict(train, test, return_scores=True)
    
    return oof_preds, test_preds, study.best_value, best_params


# %% [4-2] CatBoost Optuna
def optimize_catboost(n_trials=None):
    """CatBoost 최적화"""
    if n_trials is None:
        n_trials = TRIALS_CONFIG["CatBoost"]
    
    print(f"\n{'='*80}")
    print(f"🔍 CatBoost 최적화 ({n_trials} trials)")
    print(f"{'='*80}")
    
    def objective(trial):
        params = {
            # [핵심] CatBoost는 트리를 많이 지어야 성능이 나옴
            "iterations": trial.suggest_int("iterations", 2000, 5000), 
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            "depth": trial.suggest_int("depth", 6, 10),  # 8 -> 10 (깊은 트리 허용)
            
            # 규제 및 구조
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 30.0, log=True),
            "border_count": trial.suggest_int("border_count", 64, 254), # GPU면 254까지 가능
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 10.0), # 베이지안 부트스트랩
            "random_seed": RANDOM_STATE,
            "verbose": False,
        }
        target_device = MODEL_DEVICE_CONFIG["CatBoost"]
        if target_device == "cuda":
            params["task_type"] = "GPU"
            params["devices"] = "0"
        else:
            params["task_type"] = "CPU"
            
        model = CatBoostClassifier(**params) if TASK_TYPE == "classification" else CatBoostRegressor(**params)
        # Objective 안 model_name 추가
        wrapper = UniversalCVWrapper(model=model, model_name="CatBoost", task_type=TASK_TYPE, n_splits=CV_SPLITS, random_state=RANDOM_STATE, encoding="catboost", target_col=TARGET)
        oof_preds, _ = wrapper.fit_predict(train, test)
        
        from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error, r2_score
        
        if TASK_TYPE == "classification":
            if METRIC == "accuracy":
                return accuracy_score(train[TARGET], (oof_preds >= 0.5).astype(int))
            elif METRIC == "roc_auc":
                return roc_auc_score(train[TARGET], oof_preds)
            elif METRIC == "f1":
                return f1_score(train[TARGET], (oof_preds >= 0.5).astype(int))
        else:
            if METRIC == "rmse":
                return np.sqrt(mean_squared_error(train[TARGET], oof_preds))
            elif METRIC == "mae":
                return mean_absolute_error(train[TARGET], oof_preds)
            elif METRIC == "r2":
                return r2_score(train[TARGET], oof_preds)
    
    direction = "maximize" if METRIC in ["accuracy", "roc_auc", "f1", "r2"] else "minimize"
    study = optuna.create_study(direction=direction, sampler=TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials,timeout=OPTUNA_TIMEOUT, show_progress_bar=True)
    
    best_params = study.best_params
    best_params["verbose"] = False
    print(f"\n✨ Best {METRIC.upper()}: {study.best_value:.4f}")
    
    # [수정됨] 최종 학습 시 model_name="CatBoost" 전달
    final_model = CatBoostClassifier(**best_params) if TASK_TYPE == "classification" else CatBoostRegressor(**best_params)
    wrapper = UniversalCVWrapper(model=final_model, model_name="CatBoost", task_type=TASK_TYPE, n_splits=CV_SPLITS, random_state=RANDOM_STATE, encoding="catboost", target_col=TARGET)
    oof_preds, test_preds, _ = wrapper.fit_predict(train, test, return_scores=True)
    
    return oof_preds, test_preds, study.best_value, best_params


# %% [4-3] XGBoost Optuna
def optimize_xgboost(n_trials=None):
    """XGBoost 최적화"""
    if n_trials is None:
        n_trials = TRIALS_CONFIG["XGBoost"]

    print(f"\n{'='*80}")
    print(f"🔍 XGBoost 최적화 ({n_trials} trials)")
    print(f"{'='*80}")
    
    def objective(trial):
        params = {
            # [핵심] 충분한 학습량 보장
            "n_estimators": trial.suggest_int("n_estimators", 2000, 4000),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 6, 12), # 깊이 확장
            
            # 파라미터 미세 조정
            "min_child_weight": trial.suggest_int("min_child_weight", 2, 20),
            "gamma": trial.suggest_float("gamma", 1e-6, 5.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 50.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 50.0, log=True),
            
            "random_state": RANDOM_STATE,
            "verbosity": 0,
            # GPU 사용 시 tree_method="gpu_hist" 필수
        }

        target_device = MODEL_DEVICE_CONFIG["XGBoost"]
        if target_device == "cuda":
            params["tree_method"] = "gpu_hist"
            params["device"] = "cuda"
        else:
            params["tree_method"] = "hist"
            params["device"] = "cpu"

        model = XGBClassifier(**params) if TASK_TYPE == "classification" else XGBRegressor(**params)
        # Objective 안 model_name 추가
        wrapper = UniversalCVWrapper(model=model, model_name="XGBoost", task_type=TASK_TYPE, n_splits=CV_SPLITS, random_state=RANDOM_STATE, encoding=ENCODING, target_col=TARGET)
        oof_preds, _ = wrapper.fit_predict(train, test)

        from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error, r2_score

        if TASK_TYPE == "classification":
            if METRIC == "accuracy":
                return accuracy_score(train[TARGET], (oof_preds >= 0.5).astype(int))
            elif METRIC == "roc_auc":
                return roc_auc_score(train[TARGET], oof_preds)
            elif METRIC == "f1":
                return f1_score(train[TARGET], (oof_preds >= 0.5).astype(int))
        else:
            if METRIC == "rmse":
                return np.sqrt(mean_squared_error(train[TARGET], oof_preds))
            elif METRIC == "mae":
                return mean_absolute_error(train[TARGET], oof_preds)
            elif METRIC == "r2":
                return r2_score(train[TARGET], oof_preds)

    direction = "maximize" if METRIC in ["accuracy", "roc_auc", "f1", "r2"] else "minimize"
    study = optuna.create_study(direction=direction, sampler=TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials,timeout=OPTUNA_TIMEOUT, show_progress_bar=True)

    best_params = study.best_params
    best_params["verbosity"] = 0
    print(f"\n✨ Best {METRIC.upper()}: {study.best_value:.4f}")

    # [수정됨] 최종 학습 시 model_name="XGBoost" 전달
    final_model = XGBClassifier(**best_params) if TASK_TYPE == "classification" else XGBRegressor(**best_params)
    wrapper = UniversalCVWrapper(model=final_model, model_name="XGBoost", task_type=TASK_TYPE, n_splits=CV_SPLITS, random_state=RANDOM_STATE, encoding=ENCODING, target_col=TARGET)
    oof_preds, test_preds, _ = wrapper.fit_predict(train, test, return_scores=True)

    return oof_preds, test_preds, study.best_value, best_params


# %% [4-4] RandomForest Optuna
def optimize_rf(n_trials=None):
    """RandomForest 최적화"""
    if n_trials is None:
        n_trials = TRIALS_CONFIG["RF"]
    
    print(f"\n{'='*80}")
    print(f"🔍 RandomForest 최적화 ({n_trials} trials)")
    print(f"{'='*80}")
    
    def objective(trial):
        params = {
            # [핵심] 트리가 많아야 앙상블 효과 극대화 (최소 300)
            "n_estimators": trial.suggest_int("n_estimators", 300, 1000), 
            
            # 깊이 제한을 풀어버림 (RF는 과적합에 강함)
            "max_depth": trial.suggest_categorical("max_depth", [None, 15, 20, 25, 30]),
            
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        }
        
        model = RandomForestClassifier(**params) if TASK_TYPE == "classification" else RandomForestRegressor(**params)
        # model_name="RF" 추가
        wrapper = UniversalCVWrapper(model=model, model_name="RF", task_type=TASK_TYPE, n_splits=CV_SPLITS, random_state=RANDOM_STATE, encoding=ENCODING, target_col=TARGET)
        oof_preds, _ = wrapper.fit_predict(train, test)
        
        from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error, r2_score
        
        if TASK_TYPE == "classification":
            if METRIC == "accuracy":
                return accuracy_score(train[TARGET], (oof_preds >= 0.5).astype(int))
            elif METRIC == "roc_auc":
                return roc_auc_score(train[TARGET], oof_preds)
            elif METRIC == "f1":
                return f1_score(train[TARGET], (oof_preds >= 0.5).astype(int))
        else:
            if METRIC == "rmse":
                return np.sqrt(mean_squared_error(train[TARGET], oof_preds))
            elif METRIC == "mae":
                return mean_absolute_error(train[TARGET], oof_preds)
            elif METRIC == "r2":
                return r2_score(train[TARGET], oof_preds)
    
    direction = "maximize" if METRIC in ["accuracy", "roc_auc", "f1", "r2"] else "minimize"
    study = optuna.create_study(direction=direction, sampler=TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials, timeout=OPTUNA_TIMEOUT,show_progress_bar=True)
    
    best_params = study.best_params
    best_params["n_estimators"] = 300  # 혹은 1000
    best_params["n_jobs"] = -1
    print(f"\n✨ Best {METRIC.upper()}: {study.best_value:.4f}")
    
    # [수정됨] 최종 학습 시 model_name="RF" 전달
    final_model = RandomForestClassifier(**best_params) if TASK_TYPE == "classification" else RandomForestRegressor(**best_params)
    wrapper = UniversalCVWrapper(model=final_model, model_name="RF", task_type=TASK_TYPE, n_splits=CV_SPLITS, random_state=RANDOM_STATE, encoding=ENCODING, target_col=TARGET)
    oof_preds, test_preds, _ = wrapper.fit_predict(train, test, return_scores=True)
    
    return oof_preds, test_preds, study.best_value, best_params


# %% [4-5] ExtraTrees Optuna
def optimize_et(n_trials=None):
    """ExtraTrees 최적화"""
    if n_trials is None:
        n_trials = TRIALS_CONFIG["ET"]

    print(f"\n{'='*80}")
    print(f"🔍 ExtraTrees 최적화 ({n_trials} trials)")
    print(f"{'='*80}")

    def objective(trial):
        params = {
            # [핵심] 트리가 많아야 앙상블 효과 극대화 (최소 300)
            "n_estimators": trial.suggest_int("n_estimators", 300, 1000), 
            
            # 깊이 제한을 풀어버림 (RF는 과적합에 강함)
            "max_depth": trial.suggest_categorical("max_depth", [None, 15, 20, 25, 30]),
            
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        }

        model = ExtraTreesClassifier(**params) if TASK_TYPE == "classification" else ExtraTreesRegressor(**params)
        # model_name="ET" 추가
        wrapper = UniversalCVWrapper(model=model, model_name="ET", task_type=TASK_TYPE, n_splits=CV_SPLITS, random_state=RANDOM_STATE, encoding=ENCODING, target_col=TARGET)
        oof_preds, _ = wrapper.fit_predict(train, test)

        from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error, r2_score

        if TASK_TYPE == "classification":
            if METRIC == "accuracy":
                return accuracy_score(train[TARGET], (oof_preds >= 0.5).astype(int))
            elif METRIC == "roc_auc":
                return roc_auc_score(train[TARGET], oof_preds)
            elif METRIC == "f1":
                return f1_score(train[TARGET], (oof_preds >= 0.5).astype(int))
        else:
            if METRIC == "rmse":
                return np.sqrt(mean_squared_error(train[TARGET], oof_preds))
            elif METRIC == "mae":
                return mean_absolute_error(train[TARGET], oof_preds)
            elif METRIC == "r2":
                return r2_score(train[TARGET], oof_preds)

    direction = "maximize" if METRIC in ["accuracy", "roc_auc", "f1", "r2"] else "minimize"
    study = optuna.create_study(direction=direction, sampler=TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials,timeout=OPTUNA_TIMEOUT, show_progress_bar=True)

    best_params = study.best_params
    best_params["n_estimators"] = 300  # 혹은 1000
    best_params["n_jobs"] = -1
    print(f"\n✨ Best {METRIC.upper()}: {study.best_value:.4f}")

    # [수정됨] 최종 학습 시 model_name="ET" 전달
    final_model = ExtraTreesClassifier(**best_params) if TASK_TYPE == "classification" else ExtraTreesRegressor(**best_params)
    wrapper = UniversalCVWrapper(model=final_model, model_name="ET", task_type=TASK_TYPE, n_splits=CV_SPLITS, random_state=RANDOM_STATE, encoding=ENCODING, target_col=TARGET)
    oof_preds, test_preds, _ = wrapper.fit_predict(train, test, return_scores=True)

    return oof_preds, test_preds, study.best_value, best_params


# %% [4-6] MLP Optuna
def optimize_mlp(n_trials=None):
    """MLP 최적화"""
    if n_trials is None:
        n_trials = TRIALS_CONFIG["MLP"]
    
    print(f"\n{'='*80}")
    print(f"🔍 MLP 최적화 ({n_trials} trials)")
    print(f"{'='*80}")
    
    def objective(trial):
        # [핵심] 첫 번째 레이어를 넓게 가져감
        l0 = trial.suggest_int("n_units_l0", 64, 512)
        l1 = trial.suggest_int("n_units_l1", 32, 256)
        
        params = {
            "hidden_layer_sizes": (l0, l1),
            "activation": "relu", # relu가 국룰
            "solver": "adam",
            
            # 규제 강화 (넓어진 만큼 규제가 필요)
            "alpha": trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
            "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 5e-3, log=True),
            "batch_size": trial.suggest_categorical("batch_size", [128, 256, 512]),
            
            "max_iter": 500, # 300 -> 500 (충분한 에폭)
            "early_stopping": True, # 필수
            "n_iter_no_change": 10,
            "random_state": RANDOM_STATE,
        }
        
        model = MLPClassifier(**params) if TASK_TYPE == "classification" else MLPRegressor(**params)
        # model_name="MLP" 추가
        wrapper = UniversalCVWrapper(model=model, model_name="MLP", task_type=TASK_TYPE, n_splits=CV_SPLITS, random_state=RANDOM_STATE, encoding=ENCODING, scaling=True, target_col=TARGET)
        oof_preds, _ = wrapper.fit_predict(train, test)
        
        from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error, r2_score
        
        if TASK_TYPE == "classification":
            if METRIC == "accuracy":
                return accuracy_score(train[TARGET], (oof_preds >= 0.5).astype(int))
            elif METRIC == "roc_auc":
                return roc_auc_score(train[TARGET], oof_preds)
            elif METRIC == "f1":
                return f1_score(train[TARGET], (oof_preds >= 0.5).astype(int))
        else:
            if METRIC == "rmse":
                return np.sqrt(mean_squared_error(train[TARGET], oof_preds))
            elif METRIC == "mae":
                return mean_absolute_error(train[TARGET], oof_preds)
            elif METRIC == "r2":
                return r2_score(train[TARGET], oof_preds)
    
    direction = "maximize" if METRIC in ["accuracy", "roc_auc", "f1", "r2"] else "minimize"
    study = optuna.create_study(direction=direction, sampler=TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials,timeout=OPTUNA_TIMEOUT, show_progress_bar=True)
    
    best_params = study.best_params
    print(f"\n✨ Best {METRIC.upper()}: {study.best_value:.4f}")
    
    hidden_layers = (
    best_params.pop("n_units_l0"),
    best_params.pop("n_units_l1")
    )
    best_params["hidden_layer_sizes"] = hidden_layers
    best_params["max_iter"] = 500
    best_params["random_state"] = RANDOM_STATE
    
    # [수정됨] 최종 학습 시 model_name="MLP" 전달
    final_model = MLPClassifier(**best_params) if TASK_TYPE == "classification" else MLPRegressor(**best_params)
    wrapper = UniversalCVWrapper(model=final_model, model_name="MLP", task_type=TASK_TYPE, n_splits=CV_SPLITS, random_state=RANDOM_STATE, encoding=ENCODING, scaling=True, target_col=TARGET)
    oof_preds, test_preds, _ = wrapper.fit_predict(train, test, return_scores=True)
    
    return oof_preds, test_preds, study.best_value, best_params


# %% [4-7] Linear Model Optuna
def optimize_linear(n_trials=None):
    """Linear Model 최적화"""
    if n_trials is None:
        n_trials = TRIALS_CONFIG["Linear"]
    
    print(f"\n{'='*80}")
    print(f"🔍 Linear Model 최적화 ({n_trials} trials)")
    print(f"{'='*80}")
    
    def objective(trial):
        alpha = trial.suggest_float("alpha", 1e-3, 100.0, log=True)
        
        if TASK_TYPE == "classification":
            model = LogisticRegression(C=1/alpha, max_iter=1000, random_state=RANDOM_STATE)
        else:
            reg_type = trial.suggest_categorical("reg_type", ["Ridge", "Lasso"])
            if reg_type == "Ridge":
                model = Ridge(alpha=alpha, random_state=RANDOM_STATE)
            else:
                model = Lasso(alpha=alpha, random_state=RANDOM_STATE)
        
        # model_name="Linear" 추가
        wrapper = UniversalCVWrapper(model=model, model_name="Linear", task_type=TASK_TYPE, n_splits=CV_SPLITS, random_state=RANDOM_STATE, encoding=ENCODING, scaling=True, target_col=TARGET)
        oof_preds, _ = wrapper.fit_predict(train, test)
        
        from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error, r2_score
        
        if TASK_TYPE == "classification":
            if METRIC == "accuracy":
                return accuracy_score(train[TARGET], (oof_preds >= 0.5).astype(int))
            elif METRIC == "roc_auc":
                return roc_auc_score(train[TARGET], oof_preds)
            elif METRIC == "f1":
                return f1_score(train[TARGET], (oof_preds >= 0.5).astype(int))
        else:
            if METRIC == "rmse":
                return np.sqrt(mean_squared_error(train[TARGET], oof_preds))
            elif METRIC == "mae":
                return mean_absolute_error(train[TARGET], oof_preds)
            elif METRIC == "r2":
                return r2_score(train[TARGET], oof_preds)
    
    direction = "maximize" if METRIC in ["accuracy", "roc_auc", "f1", "r2"] else "minimize"
    study = optuna.create_study(direction=direction, sampler=TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials,timeout=OPTUNA_TIMEOUT, show_progress_bar=True)
    
    best_params = study.best_params
    print(f"\n✨ Best {METRIC.upper()}: {study.best_value:.4f}")
    
    alpha = best_params["alpha"]
    if TASK_TYPE == "classification":
        final_model = LogisticRegression(C=1/alpha, max_iter=1000, random_state=RANDOM_STATE)
    else:
        if best_params.get("reg_type") == "Ridge":
            final_model = Ridge(alpha=alpha, random_state=RANDOM_STATE)
        else:
            final_model = Lasso(alpha=alpha, random_state=RANDOM_STATE)
    
    # [수정됨] 최종 학습 시 model_name="Linear" 전달
    wrapper = UniversalCVWrapper(model=final_model, model_name="Linear", task_type=TASK_TYPE, n_splits=CV_SPLITS, random_state=RANDOM_STATE, encoding=ENCODING, scaling=True, target_col=TARGET)
    oof_preds, test_preds, _ = wrapper.fit_predict(train, test, return_scores=True)
    
    return oof_preds, test_preds, study.best_value, best_params


# %% [4-9] TabM : 기본 모델(D) + 수동 Optuna)
def optimize_tabm(n_trials=None):
    # HPO가 안 붙은 '순수 기본 모델'을 가져옵니다.
    # PyTabKit 구조상 보통 pytabkit.models.sklearn.sklearn_interfaces 안에 있습니다.
    try:
        from pytabkit.models.sklearn.sklearn_interfaces import TabM_D_Classifier, TabM_D_Regressor
    except ImportError:
        print("⚠️ TabM 기본 모델을 찾을 수 없습니다.")
        return np.zeros(len(train)), np.zeros(len(test)), 0, {}

    if n_trials is None: 
        n_trials = TRIALS_CONFIG.get("TabM", 20)

    print(f"\n{'='*80}\n🔍 TabM 수동 튜닝 ({n_trials} trials)\n{'='*80}")

    def objective(trial):
        # 1. 사용자가 직접 정의한 하이퍼파라미터 공간
        params = {
            "device": DEVICE,
            "random_state": RANDOM_STATE,
            "batch_size": 256,   
            "n_epochs": 50,      
            "verbosity": 0,       # 🚨 수정됨: verbose -> verbosity
            
            # TabM 핵심 튜닝
            "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
            "dropout": trial.suggest_float("dropout", 0.0, 0.4),
            "tabm_k": trial.suggest_int("k", 8, 16),  # 🚨 수정됨: k -> tabm_k (Optuna 변수명은 k로 유지해도 됨)
        }
        
        # 2. HPO 기능이 없는 순수 모델 생성
        model = TabM_D_Regressor(**params) if TASK_TYPE == "regression" else TabM_D_Classifier(**params)
        
        # 3. 교차 검증
        wrapper = UniversalCVWrapper(
            model=model, model_name="TabM_D", task_type=TASK_TYPE, n_splits=CV_SPLITS, 
            random_state=RANDOM_STATE, encoding=ENCODING, scaling=True, target_col=TARGET
        )
        oof_preds, _ = wrapper.fit_predict(train, test)
        
        if TASK_TYPE == "classification":
            return roc_auc_score(train[TARGET], oof_preds)
        else:
            return np.sqrt(mean_squared_error(train[TARGET], oof_preds))

    # Optuna 수행
    direction = "maximize" if METRIC in ["accuracy", "roc_auc", "f1", "r2"] else "minimize"
    study = optuna.create_study(direction=direction, sampler=TPESampler(seed=RANDOM_STATE))
    study.optimize(objective,timeout=OPTUNA_TIMEOUT, n_trials=n_trials)

    print(f"✅ TabM Best: {study.best_params}")
    
    # 베스트 파라미터로 최종 학습 (에폭 늘리기)
    best_params = study.best_params
    best_params.update({"device": DEVICE, "n_epochs": 200, "verbosity": 1, "batch_size": 256})
    
    final_model = TabM_D_Regressor(**best_params) if TASK_TYPE == "regression" else TabM_D_Classifier(**best_params)
    wrapper = UniversalCVWrapper(
        model=final_model, model_name="TabM_D", task_type=TASK_TYPE, n_splits=CV_SPLITS, 
        random_state=RANDOM_STATE, encoding=ENCODING, scaling=True, target_col=TARGET
    )
    oof_preds, test_preds, _ = wrapper.fit_predict(train, test, return_scores=True)
    
    return oof_preds, test_preds, study.best_value, best_params


# %% [4-10] RealMLP : TD(Tuned Defaults) 모델 사용 (튜닝 불필요)
def optimize_realmlp(n_trials=None):
    try:
        from pytabkit.models.sklearn.sklearn_interfaces import RealMLP_TD_Classifier, RealMLP_TD_Regressor
    except ImportError:
        print("⚠️ RealMLP TD 모델을 찾을 수 없습니다.")
        return np.zeros(len(train)), np.zeros(len(test)), 0, {}

    print(f"\n{'='*80}\n🔍 RealMLP-TD 실행 (Tuned Defaults 사용, 별도 튜닝 없음)\n{'='*80}")

    # TD 모델은 파라미터 고민할 필요 없이 설정만 잡아주면 됨
    params = {
        "device": DEVICE,
        "random_state": RANDOM_STATE,
        "n_epochs": 200,      # 충분히 학습
        "batch_size": 256,
        "verbosity": 1,
    }

    model = RealMLP_TD_Regressor(**params) if TASK_TYPE == "regression" else RealMLP_TD_Classifier(**params)
    
    # 바로 학습
    wrapper = UniversalCVWrapper(
        model=model, model_name="RealMLP_TD", task_type=TASK_TYPE, n_splits=CV_SPLITS, 
        random_state=RANDOM_STATE, encoding=ENCODING, scaling=True, target_col=TARGET
    )
    
    # 시간 측정 및 학습
    oof_preds, test_preds, _ = wrapper.fit_predict(train, test, return_scores=True)

    if TASK_TYPE == "classification":
        score = roc_auc_score(train[TARGET], oof_preds)
    else:
        score = np.sqrt(mean_squared_error(train[TARGET], oof_preds))
        
    print(f"✅ RealMLP-TD Score: {score:.5f}")

    # 형식을 맞추기 위해 dummy params 리턴
    return oof_preds, test_preds, score, {"model": "RealMLP_TD", "note": "Used Pre-tuned Defaults"}

# %% [5] 모델 실행 (즉시 저장 기능 추가)
import os

oof_dict = {}
test_dict = {}
scores = {}

# 1. 모델별 최적화 함수 매핑 
model_functions = {
    "LGBM": optimize_lgbm,
    "CatBoost": optimize_catboost,
    "XGBoost": optimize_xgboost,
    "RF": optimize_rf,
    "ET": optimize_et,
    "MLP": optimize_mlp,
    "Linear": optimize_linear,
    "TabM": optimize_tabm,
    "RealMLP": optimize_realmlp,
}

# 2. 설정된 모델만 반복 실행
print(f"\n{'='*80}")
print(f"🚀 활성화된 모델 순차 실행 시작 (완료 즉시 자동 저장)")
print(f"{'='*80}")

for model_name, run_func in model_functions.items():
    # 스위치가 켜져 있는지 확인
    if ACTIVE_MODELS.get(model_name, False):
        try:
            print(f"\n▶️ Starting {model_name}...")
            
            # 함수 실행 (oof, test_pred, score, params 순서로 리턴됨)
            curr_oof, curr_test, curr_score, curr_params = run_func()
            
            # 결과가 정상적으로 나왔다면
            if curr_score != 0: 
                # 1. 메모리에 저장 (나중에 요약용)
                oof_dict[model_name] = curr_oof
                test_dict[model_name] = curr_test
                scores[model_name] = curr_score
                
                # ====================================================
                # 💾 [핵심] 모델 끝나자마자 바로 CSV 저장 (Safety Save)
                # ====================================================
                
                # 출력 디렉토리 생성
                OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
                
                # (1) Test Prediction (제출용) 저장
                sub_df = pd.DataFrame()
                sub_df[ID_COL] = test[ID_COL].values
                sub_df[TARGET] = curr_test
                sub_filename = OUTPUT_DIR / f"submission_{model_name}_{USER_NAME}.csv"
                sub_df.to_csv(sub_filename, index=False)
                
                # (2) OOF Prediction (앙상블용) 저장
                oof_df = pd.DataFrame()
                oof_df[ID_COL] = train[ID_COL].values
                oof_df[TARGET] = curr_oof
                oof_filename = OUTPUT_DIR / f"oof_{model_name}_{USER_NAME}.csv"
                oof_df.to_csv(oof_filename, index=False)
                
                print(f"✅ [Saved Instantly] {sub_filename.name}, {oof_filename.name}")
                # ====================================================
                
        except Exception as e:
            print(f"⚠️ Error running {model_name}: {e}")
            continue
    else:
        print(f"⏭️ Skipping {model_name} (Disabled)")

print(f"\n✅ 모든 모델 실행 완료")

# [8] 결과 요약 출력
if scores:
    print(f"\n{'='*80}")
    print(f"📊 전체 결과 요약")
    print(f"{'='*80}")

    summary = pd.DataFrame({
        "Model": list(scores.keys()),
        f"{METRIC.upper()}": list(scores.values())
    })

    if METRIC in ["rmse", "mae"]:
        summary = summary.sort_values(METRIC.upper(), ascending=True)
    else:
        summary = summary.sort_values(METRIC.upper(), ascending=False)

    print(summary.to_string(index=False))
    print(f"\n🏆 Best Model: {summary.iloc[0]['Model']}")

[I 2025-12-09 11:49:46,984] A new study created in memory with name: no-name-bcb4f822-e7cb-48e4-82a4-78c0e7c6ae05


✅ 모델 및 최적화 라이브러리 import 완료

🚀 활성화된 모델 순차 실행 시작 (완료 즉시 자동 저장)

▶️ Starting LGBM...

🔍 LGBM 최적화 (10000 trials)


  0%|          | 0/10000 [00:00<?, ?it/s]

🚀 [CV Start] Model: LGBM | Device: CPU | Enc: target (sm=10.00) | Scaler: standard
  Fold 1 | Acc: 0.7017 | AUC: 0.6649
  Fold 1 | Acc: 0.7017 | AUC: 0.6649
  Fold 2 | Acc: 0.7047 | AUC: 0.6709
  Fold 2 | Acc: 0.7047 | AUC: 0.6709
  Fold 3 | Acc: 0.7017 | AUC: 0.6693
  Fold 3 | Acc: 0.7017 | AUC: 0.6693
  Fold 4 | Acc: 0.7023 | AUC: 0.6613
  Fold 4 | Acc: 0.7023 | AUC: 0.6613
  Fold 5 | Acc: 0.7047 | AUC: 0.6758

✨ LGBM CV Complete.
[I 2025-12-09 11:49:52,340] Trial 0 finished with value: 0.6684149756854542 and parameters: {'n_estimators': 3123, 'learning_rate': 0.0862735828664018, 'num_leaves': 196, 'max_depth': 13, 'min_child_samples': 24, 'subsample': 0.662397808134481, 'colsample_bytree': 0.6232334448672797, 'reg_alpha': 8.635984878977442, 'reg_lambda': 0.2665240604681077}. Best is trial 0 with value: 0.6684149756854542.
🚀 [CV Start] Model: LGBM | Device: CPU | Enc: target (sm=10.00) | Scaler: standard
  Fold 5 | Acc: 0.7047 | AUC: 0.6758

✨ LGBM CV Complete.
[I 2025-12-09 11:49:52

[I 2025-12-09 11:50:26,675] A new study created in memory with name: no-name-02379e53-c25a-45ec-8feb-7934af022a80


  Fold 5 | Acc: 0.7012 | AUC: 0.6727

✨ LGBM CV Complete.
✅ [Saved Instantly] submission_LGBM_semin.csv, oof_LGBM_semin.csv

▶️ Starting CatBoost...

🔍 CatBoost 최적화 (10000 trials)


  0%|          | 0/10000 [00:00<?, ?it/s]

🚀 [CV Start] Model: CatBoost | Device: CPU | Enc: catboost (sm=10.00) | Scaler: standard


### 5-2. 최적화 함수 및 실행
모든 모델에 대한 Optuna 최적화 함수와 실행 코드